# BSS

In [22]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression

In [23]:
def backward_stepwise_selection(X,y,feature_names=None):
    X=np.asarray(X, dtype=float)
    y = np.asarray(y,dtype=float).ravel()
    
    n,p = X.shape
    
    TSS = np.sum((y - np.mean(y))**2)
    def evaluate(features):
        
        if len(features) == 0:
            y_pred = np.full_like(y, np.mean(y))
        else:
            
            model = LinearRegression()
            model.fit(X[:, features],y)
            y_pred = model.predict(X[:, features])
        
        RSS = np.sum((y - y_pred) ** 2)
        R2 = 1 - RSS / TSS
        
        q = len(features)
        
        if n - q - 1 > 0:
            adjusted_R2 = 1 - ((RSS / (n-q-1)) / (TSS/(n-1)))
        
        else:
            adjusted_R2 = np.nan
        return RSS, R2, adjusted_R2
    
    selected = list(range(p))
    history = []
    
    RSS, R2, adj_R2 = evaluate(selected)
    
    history.append({
        "Step": 0,
        "removed feature": "None",
        "Features": [feature_names[i] for i in selected],
        "Number of Features": len(selected),
        "RSS": RSS,
        "R2": R2,
        "Adjusted R2": adj_R2
    })
    
    step = 0
    
    while len(selected) > 1:
        
        best_candidate = None
        
        best_metrics = None
        best_adj_R2 = -np.inf
        
        for feature in selected:
            
            candidate = [ f for f in selected if f!= feature]
            RSS_c, R2_c, adj_R2_c = evaluate(candidate)
            
            if adj_R2_c > best_adj_R2:
                best_adj_R2 = adj_R2_c
                best_candidate = (feature, candidate)
                best_metrics = (RSS_c, R2_c, adj_R2_c)
        
        if best_adj_R2 <= adj_R2 + 1e-12:
            break
            
        removed_feature, selected = best_candidate
        RSS, R2, adj_R2 = best_metrics
        step += 1
        
        history.append({
            "Step": step,
            "Removed feature": feature_names[removed_feature],
            "Features": [feature_names[i] for i in selected],
            "Number of features": len(selected),
            "RSS": RSS,
            "R2": R2,
            "Adjusted_R2": adj_R2
       
        })
        
    results_df = pd.DataFrame(history)
    
    selected_features = [feature_names[i] for i in selected]
    
    return selected_features, results_df

In [24]:
from sklearn.datasets import load_diabetes
data = load_diabetes()

X = data.data
y = data.target

feature_names = data.feature_names

print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])
print("feature names:", feature_names)

Number of samples: 442
,Number of features: 10
,feature names: ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']


In [25]:
selected_features, results_df = (backward_stepwise_selection(X,y,feature_names))

print("Selected features",selected_features)
print("Number of features :",len(selected_features))
print("Selection History")

print(results_df[[ "Step",
            "Removed feature",
            "Features" ,
            "Number of features",
            "RSS",
            "R2",
            "Adjusted_R2"]].to_string(index=False))

Selected features ['sex', 'bmi', 'bp', 's1', 's2', 's4', 's5', 's6']
,Number of features : 8
,Selection History
, Step Removed feature                                    Features  Number of features          RSS       R2  Adjusted_R2
,    0             NaN [age, sex, bmi, bp, s1, s2, s3, s4, s5, s6]                 NaN 1.263983e+06 0.517749          NaN
,    1             age      [sex, bmi, bp, s1, s2, s3, s4, s5, s6]                 9.0 1.264066e+06 0.517718     0.507670
,    2              s3          [sex, bmi, bp, s1, s2, s4, s5, s6]                 8.0 1.264712e+06 0.517471     0.508556
